# Repeat-features circos: tandem + interspersed repeats, HORs, 349-bp centromeric repeats

Circos plot of the degu genome. Radial order (outside → inside):
1. **Chromosomes** (r=95–100) — ticks + scaffold junctions
2. **349-bp centromeric tandem repeats** (r=94–84) — TRF
3. **HORs** (r=83–73) — centroAnno higher-order repeats
4. **Tandem-repeat density** (r=72–61)
5. **Interspersed-repeat density** (r=60–50)

Cleaned from `figure/circos-plot/repeat-features/pyCircularize.ipynb` (tutorial and
exploratory cells removed; input paths generalized via `PROJ_ROOT`).


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pycirclize import Circos
from Bio.SeqFeature import SeqFeature, FeatureLocation

%matplotlib inline


In [ ]:
# ============================================================================
# Load data
# ============================================================================

# Chromosome sizes (.fai)
fai = pd.read_csv(
    f"{PROJ_ROOT}/data/denovo_OctDegus_genome/041425-assembly/hifiasm-041425-assembly-mitoFiltered-scaffolded-curated-masked-chrNameAssigned/assembly_final.sorted.headerRenamed.chrAssigned.hardMasked.mito.fasta.fai",
    sep="\t", header=None,
    names=["chr", "length", "offset", "linebases", "linewidth"],
)
karyotype = fai[fai["chr"].str.contains("chr")][["chr", "length"]].reset_index(drop=True)
chr_sizes = dict(zip(karyotype["chr"], karyotype["length"]))

# 349-bp centromeric tandem repeats (TRF)
df_cent = pd.read_csv(
    f"{PROJ_ROOT}/code/command-line-script/genome-annotation/trf-tandem-repeat/349peak_repeat_1millionBpMin.tsv",
    sep="\t", index_col=False,
)

# Higher-order repeats (centroAnno)
hor_bed = pd.read_csv(
    f"{PROJ_ROOT}/output/outputs-from-centraAnno/hifiasm-0414/cautils-chrOnly/HORs.bed",
    sep="\t", header=0,
    dtype={
        "Sequence_Name": "string", "HOR_Name": "string",
        "Start_Position": "int64", "End_Position": "int64",
        "Num_monomers": "int64", "HOR_len": "int64", "Span_len": "int64",
    },
    names=["Sequence_Name", "HOR_Name", "Start_Position", "End_Position",
           "Num_monomers", "HOR_len", "Span_len"],
    na_values=["."], keep_default_na=False,
)

# Interspersed-repeat density (1 Mb windows, 500 kb step; precomputed)
final_density = pd.read_csv(
    f"{PROJ_ROOT}/figure/circos-plot/repeat-features/assembly_final.sorted.headerRenamed.repeatDensity.1mb_500kStep_rollingWindow.tsv",
    sep="\t",
)

# Tandem-repeat density (precomputed)
final_density_tan = pd.read_csv(
    f"{PROJ_ROOT}/figure/circos-plot/repeat-features/assembly_final.sorted.headerRenamed.tanRepeatDensity.merged.tsv",
    sep="\t",
)

# Scaffold junctions (contig boundaries within chromosomes)
junction_bed = pd.read_csv(
    f"{PROJ_ROOT}/figure/circos-plot/feature-overview/agp_final_contig2scaffold.bed",
    sep="\t", header=0,
    dtype={"chr": "string", "start": "int64", "end": "int64"},
    names=["chr", "start", "end"],
    na_values=["."], keep_default_na=False,
)

print(f"{len(karyotype)} chromosomes")
print(f"349-bp repeats: {df_cent.shape}, HORs: {hor_bed.shape}, junctions: {junction_bed.shape}")


In [ ]:
# ============================================================================
# Plot the circos
# ============================================================================
y_max = 1
circos = Circos(sectors=chr_sizes, start=30, end=359, space=2, endspace=False)

for i, sector in enumerate(circos.sectors):
    # 2) 349-bp centromeric tandem repeats
    centromere_track = sector.add_track((94, 84), r_pad_ratio=0.1)
    features_this_chr = df_cent[df_cent["chromosome"] == sector.name]
    features_to_plot = [
        SeqFeature(
            FeatureLocation(int(row["start"]), int(row["end"])),
            type="repeat",
            qualifiers={
                "match_percent": row["match_percent"],
                "repeat_point_relative_perc": row["repeat_point_relative_perc"],
            },
        )
        for _, row in features_this_chr.iterrows()
    ]
    centromere_track.genomic_features(features_to_plot, fc="gold", ec="goldenrod", lw=0.5, alpha=0.8)

    # 3) HOR track
    hor_track = sector.add_track((83, 73), r_pad_ratio=0.1)
    features_this_chr = hor_bed[hor_bed["Sequence_Name"] == sector.name]
    features_to_plot = [
        SeqFeature(FeatureLocation(int(row["Start_Position"]), int(row["End_Position"])))
        for _, row in features_this_chr.iterrows()
    ]
    hor_track.genomic_features(features_to_plot, fc="green", ec="green", lw=0.5, alpha=0.8)

    # 4) Tandem-repeat density
    sub_density = final_density_tan[final_density_tan["Chromosome"] == sector.name]
    if sub_density.empty:
        print(f"No tandem density data for {sector.name}")
        continue
    tan_track = sector.add_track((72, 61))
    tan_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y_scaled = np.clip(sub_density["density"].values, 0, y_max)
    tan_track.fill_between(x, y_scaled, 0, vmin=0, vmax=1, color="#0096FF", alpha=0.5)

    # 5) Interspersed-repeat density
    sub_density = final_density[final_density["Chromosome"] == sector.name]
    if sub_density.empty:
        print(f"No interspersed density data for {sector.name}")
        continue
    density_track = sector.add_track((60, 50))
    density_track.axis(fc="none", ec="grey", lw=0.5)
    x = (sub_density["Start"] + sub_density["End"]) / 2
    y = sub_density["density"].values
    x_clipped = np.clip(x, sector.start, sector.end)
    y_norm = (y - y.min()) / (y.max() - y.min()) if y.max() > 0 else y
    density_track.fill_between(x_clipped, y_norm, 0, vmax=1, color="#FFB6B6", alpha=0.5, ec="none")

    # 1) Chromosome track: scaffold junctions + major/minor ticks
    junction_track = sector.add_track((95, 100))
    chr_junctions = junction_bed[junction_bed["chr"] == sector.name]
    for _, row in chr_junctions.iterrows():
        junction_track.line(
            x=[row["start"], row["end"]], y=[0, 100],
            color="black", lw=1, arc=True, ls=":",
        )

    outer = sector.add_track((95, 100))
    outer.axis(fc="none", ec="black", lw=0.5)
    outer.xticks_by_interval(
        interval=50_000_000, tick_length=3, outer=True, show_label=True,
        label_size=8, label_orientation="vertical",
        label_formatter=lambda v: f"{v/1e6:.0f} Mb", line_kws=dict(ec="grey"),
    )
    outer.xticks_by_interval(
        interval=10_000_000, tick_length=1, outer=True, show_label=False,
        line_kws=dict(ec="black", lw=0.5),
    )
    mid = (sector.start + sector.end) / 2
    sector.text(text=sector.name, x=mid, r=115, adjust_rotation=True, size=10)

    if i == 0:
        tan_track.yticks(y=[0.5, 1.0], labels=["0.5", "1"], side="left", tick_length=2, label_size=7)
        density_track.yticks(y=[0.5, 1.0], labels=["0.5", "1"], side="left", tick_length=2, label_size=7)

# Render + track labels (in the 0-deg gap)
fig = circos.plotfig()
ax = fig.axes[0]

track_labels = [
    (95, "1) Chromosomes", "black"),
    (83, "2) 349-bp tandem\n    repeats", "black"),
    (75, "3) HOR", "black"),
    (62, "4) Tandem\n    repeat", "black"),
    (48, "5) Interspersed\n    repeat", "black"),
]
for track_top, text, color in track_labels:
    ax.text(
        np.deg2rad(0), track_top + 1.5, text, rotation=0,
        ha="left", va="bottom", color=color, size=8, fontweight="bold",
    )

fig.tight_layout()
fig.show()


In [ ]:
# Save figure (PNG + SVG)
fig.savefig("degus_genome_circos_repeat_overview.png", dpi=600, bbox_inches="tight",
            transparent=False, facecolor="white")
fig.savefig("degus_genome_circos_repeat_overview.svg", bbox_inches="tight",
            transparent=False, facecolor="white")
